In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/BassamXYZ/md-net-underwater-image-enhancement /content/md-net-underwater-image-enhancement

In [ ]:
import kagglehub
path1 = kagglehub.dataset_download("satyendrapalsingh/uieb-dataset")
path2 = kagglehub.dataset_download("noureldin199/lsui-large-scale-underwater-image-dataset")
print(path1, path2)

Using Colab cache for faster access to the 'uieb-dataset' dataset.
Using Colab cache for faster access to the 'lsui-large-scale-underwater-image-dataset' dataset.
/kaggle/input/uieb-dataset /kaggle/input/lsui-large-scale-underwater-image-dataset


In [ ]:
import os, sys
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset

REPO = "/content/md-net-underwater-image-enhancement"
sys.path.append(REPO)
from model import NeuralNetwork, Trainer
from model.datasets import UnderwaterDataset, UIEBChallenging
from model.util import to_pil, load_trained_model
import uie_metrics as M

# ---------------- Settings ----------------
BASE      = "/content/drive/MyDrive"
CKPT_DIR  = f"{BASE}/checkpoints"
CKPT      = f"{CKPT_DIR}/mdnet_checkpoint_epoch_50.pth"
TEST_OUT  = f"{BASE}/tests_fixed"
CHALLENGING_DIR = f"{BASE}/challenging"

IMG_SIZE   = 350
BATCH_SIZE = 4

# A GPU is strongly recommended for training (the model also runs on CPU, slowly).
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

def make_splits():
    """Same split as the paper: LSUI 3794/485 and UIEB 800/90 (according to filename order)."""
    lsui = UnderwaterDataset(os.path.join(path2, "LSUI/input"), os.path.join(path2, "LSUI/GT"), transform)
    uieb = UnderwaterDataset(os.path.join(path1, "UIEB/raw-890"), os.path.join(path1, "UIEB/reference-890"), transform)
    assert len(lsui) >= 3794 + 485, f"LSUI: only {len(lsui)} images found"
    assert len(uieb) >= 800 + 90,   f"UIEB: only {len(uieb)} images found"
    return {
        "lsui_train": Subset(lsui, range(0, 3794)),
        "lsui_val":   Subset(lsui, range(3794, 3794 + 485)),
        "uieb_train": Subset(uieb, range(0, 800)),
        "uieb_val":   Subset(uieb, range(800, 800 + 90)),
    }

def sanity_check(model, loader):
    """Quick check: the output of an unloaded network is near-zero with many negative values."""
    x = next(iter(loader))
    x = x[0] if isinstance(x, (list, tuple)) else x
    with torch.no_grad():
        out, _ = model(x.to(device))
    out = out.cpu()
    outside = ((out < 0) | (out > 1)).float().mean().item()
    print(f"input mean {x.mean():.3f} | output mean {out.mean():.3f} | outside [0,1]: {outside:.2%}")
    if out.mean() < 0.1 or (out < 0).float().mean() > 0.2:
        print("WARNING: The output looks like an untrained network -> make sure the checkpoint was actually loaded")

---
# Section 1 — Training
- Default settings: 50 epochs, batch 4, Adam with a learning rate of 1e-4 (as in the paper).
- To resume, set the checkpoint path in `RESUME_FROM` and it will continue until `TOTAL_EPOCHS`.
- Saves a checkpoint after each epoch with the name `mdnet_checkpoint_epoch_<n>.pth` and does not overwrite the previous one, but **do not re-run the same epoch** because it will overwrite its file.

In [ ]:
splits = make_splits()
train_dataset = ConcatDataset([splits["lsui_train"], splits["uieb_train"]])
val_dataset   = ConcatDataset([splits["lsui_val"],   splits["uieb_val"]])
train_dataloader    = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
validate_dataloader = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(len(train_dataset), "train /", len(val_dataset), "val")

In [ ]:
TOTAL_EPOCHS = 50
RESUME_FROM  = None # f"{CKPT_DIR}/mdnet_checkpoint_epoch_50.pth"

model = NeuralNetwork().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
trainer = Trainer(model, optimizer, device, checkpoint_dir=CKPT_DIR)

start_epoch = 0
if RESUME_FROM:
    start_epoch = trainer.load_checkpoint(RESUME_FROM) + 1

if start_epoch >= TOTAL_EPOCHS:
    print("Training is already complete, nothing to run.")
else:
    trainer.train(
        train_loader=train_dataloader,
        val_loader=validate_dataloader,
        num_epochs=TOTAL_EPOCHS - start_epoch,
        use_supervision=True,
        save_interval=1,
        start_epoch=start_epoch,
    )

---
# Section 2 — Testing on LSUI and UIEB
Loads the checkpoint from `CKPT` and saves the original images, the output, and the reference, then computes PSNR, SSIM, UCIQE, and UIQM for each set separately, with a comparison against the raw images.

In [ ]:
splits = make_splits()
test_sets = {"lsui": splits["lsui_val"], "uieb": splits["uieb_val"]}
test_loaders = {k: DataLoader(v, batch_size=BATCH_SIZE, shuffle=False, num_workers=2) for k, v in test_sets.items()}

model = load_trained_model(CKPT, device)
sanity_check(model, test_loaders["lsui"])

In [ ]:
def run_test(model, subset, loader, name):
    for sub in ("inputs", "outputs", "references"):
        os.makedirs(f"{TEST_OUT}/{name}/{sub}", exist_ok=True)
    base, n, outside = subset.dataset, 0, 0.0
    with torch.no_grad():
        for inputs, refs in tqdm(loader, desc=f"Test {name}"):
            outs, _ = model(inputs.to(device))
            outside += ((outs < 0) | (outs > 1)).float().mean().item() * len(outs)
            for i in range(outs.shape[0]):
                stem = os.path.splitext(base.input_images[subset.indices[n]])[0]
                to_pil(inputs[i]).save(f"{TEST_OUT}/{name}/inputs/{stem}.png")
                to_pil(outs[i]).save(f"{TEST_OUT}/{name}/outputs/{stem}.png")
                to_pil(refs[i]).save(f"{TEST_OUT}/{name}/references/{stem}.png")
                n += 1
    print(f"{name}: {n} images, output values outside [0,1]: {outside / n:.3%}")

for name in ("lsui", "uieb"):
    run_test(model, test_sets[name], test_loaders[name], name)

Test lsui: 100%|██████████| 122/122 [13:58<00:00,  6.87s/it]


lsui: 485 images, output values outside [0,1]: 0.538%


Test uieb: 100%|██████████| 23/23 [02:37<00:00,  6.85s/it]

uieb: 90 images, output values outside [0,1]: 1.494%


In [ ]:
for name in ("lsui", "uieb"):
    print(f"\n=== {name.upper()} ===")
    print("-- Raw images (paper: LSUI 21.32/0.765, UIEB 19.17/0.749 PSNR/SSIM) --")
    _ = M.evaluate_folder(f"{TEST_OUT}/{name}/inputs", f"{TEST_OUT}/{name}/references")
    print("-- MD-Net --")
    rows = M.evaluate_folder(f"{TEST_OUT}/{name}/outputs", f"{TEST_OUT}/{name}/references")
    worst = sorted(rows, key=lambda r: r["PSNR"])[:5]
    print("Worst 5 images (PSNR):", [(w["name"], round(float(w["PSNR"]), 2)) for w in worst])


=== LSUI ===
-- Raw images (paper: LSUI 21.32/0.765, UIEB 19.17/0.749 PSNR/SSIM) --
Evaluated 485 images
  UCIQE : 0.4264
  UIQM  : 1.9818
  PSNR  : 13.2736
  SSIM  : 0.7118
-- MD-Net --
Evaluated 485 images
  UCIQE : 0.5582
  UIQM  : 2.6557
  PSNR  : 21.5278
  SSIM  : 0.8950
Worst 5 images (PSNR): [('981.png', 10.5), ('742.png', 11.63), ('896.png', 11.9), ('980.png', 11.91), ('915.png', 11.95)]

=== UIEB ===
-- Raw images (paper: LSUI 21.32/0.765, UIEB 19.17/0.749 PSNR/SSIM) --
Evaluated 90 images
  UCIQE : 0.5184
  UIQM  : 2.5421
  PSNR  : 17.0433
  SSIM  : 0.7653
-- MD-Net --
Evaluated 90 images
  UCIQE : 0.5957
  UIQM  : 2.8946
  PSNR  : 20.3277
  SSIM  : 0.8715
Worst 5 images (PSNR): [('869_img_.png', 12.73), ('905_img_.png', 12.91), ('86_img_.png', 13.23), ('850_img_.png', 13.29), ('912_img_.png', 13.46)]


---
# Section 3 — Testing on UIEB Challenging
Images without a reference, so only UCIQE and UIQM are computed. The raw images are compared **after resizing them to 350** because the two metrics depend on resolution.

In [ ]:
challenging_dataset = UIEBChallenging(CHALLENGING_DIR, transform=transform)
challenging_dataloader = DataLoader(challenging_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(len(challenging_dataset), "images")

model = load_trained_model(CKPT, device)
sanity_check(model, challenging_dataloader)

60 images
load_state_dict: <All keys matched successfully>
checkpoint epoch index: 49
input mean 0.345 | output mean 0.466 | outside [0,1]: 2.51%


In [ ]:
CH_OUT = f"{TEST_OUT}/challenging"
os.makedirs(CH_OUT, exist_ok=True)
k = 0
with torch.no_grad():
    for batch in tqdm(challenging_dataloader, desc="UIEB Challenging"):
        outs, _ = model(batch.to(device))
        for i in range(outs.shape[0]):
            stem = os.path.splitext(challenging_dataset.input_images[k])[0]
            to_pil(outs[i]).save(f"{CH_OUT}/{stem}.png")
            k += 1
print("saved", k, "images to", CH_OUT)

UIEB Challenging: 100%|██████████| 15/15 [00:46<00:00,  3.09s/it]

saved 60 images to /content/drive/MyDrive/tests_fixed/challenging


In [ ]:
print("-- Raw images (resized to 350) --")
_ = M.evaluate_folder(CHALLENGING_DIR, size=IMG_SIZE)
print("-- MD-Net --")
rows = M.evaluate_folder(CH_OUT)
worst = sorted(rows, key=lambda r: r["UIQM"])[:5]
print("Worst 5 images (UIQM):", [(w["name"], round(float(w["UIQM"]), 2)) for w in worst])

-- Raw images (resized to 350) --
Evaluated 60 images
  UCIQE : 0.4743
  UIQM  : 1.9206
-- MD-Net --
Evaluated 60 images
  UCIQE : 0.5599
  UIQM  : 2.3277
Worst 5 images (UIQM): [('15775.png', 0.6), ('12625.png', 0.87), ('12753.png', 1.03), ('15740.png', 1.06), ('15735.png', 1.11)]


# Additional Experiments

Sections:
- **4** Comparison with classical methods (Gray-World, CLAHE, DCP, UDCP)
- **5** LPIPS metric
- **6** Applied test (SIFT + Canny edges)

In [ ]:
import cv2, glob, io, contextlib
import pandas as pd
for need in ("TEST_OUT", "CH_OUT", "CHALLENGING_DIR", "M", "device"):
    assert need in globals(), f"Variable {need} is not defined: run the shared setup and Sections 2 and 3 first"
for d in ("lsui", "uieb"):
    for sub in ("inputs", "outputs", "references"):
        assert os.path.isdir(f"{TEST_OUT}/{d}/{sub}"), f"Missing: {TEST_OUT}/{d}/{sub} (run Section 2)"
assert os.path.isdir(CH_OUT), "Missing Challenging output (run Section 3)"

---
# Section 4 — Comparison with classical methods (without training)
Each method is applied to **the same test images** saved at size 350×350, then the same metrics are computed.
- **Gray-World:** adjusting the channel gains so their means become equal.
- **CLAHE:** adaptive contrast on the L illumination channel in Lab space.
- **GW+CLAHE:** both together.
- **DCP:** Dark Channel Prior (He et al.) with Guided Filter smoothing.
- **UDCP:** underwater version (G and B channels only).
My implementations are simplified with default parameters, so they are a reasonable baseline and not the best these methods can achieve.

In [ ]:
def read_rgb(path, size=None):
    img = cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    if size: img = cv2.resize(img, (size, size), interpolation=cv2.INTER_CUBIC)
    return img

def gray_world(img):
    x = img.astype(np.float64)
    m = x.reshape(-1, 3).mean(0) + 1e-6
    return np.clip(x * (m.mean() / m), 0, 255).astype(np.uint8)

def clahe_lab(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def gw_clahe(img):
    return clahe_lab(gray_world(img))

def _guided_filter(guide, src, r=40, eps=1e-3):
    k = (2 * r + 1, 2 * r + 1)
    mean = lambda x: cv2.boxFilter(x, -1, k)
    mI, mp = mean(guide), mean(src)
    cov = mean(guide * src) - mI * mp
    var = mean(guide * guide) - mI * mI
    a = cov / (var + eps)
    b = mp - a * mI
    return mean(a) * guide + mean(b)

def _dehaze(img, channels, omega=0.95, patch=15, t0=0.1):
    I = img.astype(np.float64) / 255.0
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    dark = cv2.erode(I[..., channels].min(axis=2), k)
    # Atmospheric light: the brightest 0.1% of the dark channel, then the highest intensity among them
    n = max(int(dark.size * 0.001), 1)
    idx = np.argsort(dark.ravel())[-n:]
    cand = I.reshape(-1, 3)[idx]
    A = cand[cand.sum(1).argmax()] + 1e-6
    t = 1.0 - omega * cv2.erode((I / A)[..., channels].min(axis=2), k)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
    t = np.clip(_guided_filter(gray, t), t0, 1.0)
    J = (I - A) / t[..., None] + A
    return np.clip(J * 255, 0, 255).astype(np.uint8)

def dcp(img):  return _dehaze(img, [0, 1, 2])
def udcp(img): return _dehaze(img, [1, 2])

CLASSICAL = {"GrayWorld": gray_world, "CLAHE": clahe_lab, "GW+CLAHE": gw_clahe, "DCP": dcp, "UDCP": udcp}

def apply_to_folder(fn, in_dir, out_dir, size=None):
    os.makedirs(out_dir, exist_ok=True)
    for p in sorted(glob.glob(f"{in_dir}/*")):
        if not p.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")): continue
        out = fn(read_rgb(p, size))
        cv2.imwrite(os.path.join(out_dir, os.path.splitext(os.path.basename(p))[0] + ".png"),
                    cv2.cvtColor(out, cv2.COLOR_RGB2BGR))

In [ ]:
CL_OUT = f"{TEST_OUT}/classical"
METHOD_DIRS = {}

for d in ("lsui", "uieb"):
    METHOD_DIRS[d] = {"Raw": f"{TEST_OUT}/{d}/inputs"}
    for name, fn in CLASSICAL.items():
        out_dir = f"{CL_OUT}/{d}/{name}"
        apply_to_folder(fn, f"{TEST_OUT}/{d}/inputs", out_dir)
        METHOD_DIRS[d][name] = out_dir
    METHOD_DIRS[d]["MD-Net"] = f"{TEST_OUT}/{d}/outputs"

raw350 = f"{TEST_OUT}/challenging_raw350"
apply_to_folder(lambda x: x, CHALLENGING_DIR, raw350, size=IMG_SIZE)
METHOD_DIRS["challenging"] = {"Raw": raw350}
for name, fn in CLASSICAL.items():
    out_dir = f"{CL_OUT}/challenging/{name}"
    apply_to_folder(fn, raw350, out_dir)
    METHOD_DIRS["challenging"][name] = out_dir
METHOD_DIRS["challenging"]["MD-Net"] = CH_OUT
print({k: list(v) for k, v in METHOD_DIRS.items()})

{'lsui': ['Raw', 'GrayWorld', 'CLAHE', 'GW+CLAHE', 'DCP', 'UDCP', 'MD-Net'], 'uieb': ['Raw', 'GrayWorld', 'CLAHE', 'GW+CLAHE', 'DCP', 'UDCP', 'MD-Net'], 'challenging': ['Raw', 'GrayWorld', 'CLAHE', 'GW+CLAHE', 'DCP', 'UDCP', 'MD-Net']}


In [ ]:
def quiet_eval(pred_dir, gt_dir=None):
    with contextlib.redirect_stdout(io.StringIO()):
        rows = M.evaluate_folder(pred_dir, gt_dir)
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0] if k != "name"}

tables = {}
for d, methods in METHOD_DIRS.items():
    gt = f"{TEST_OUT}/{d}/references" if d != "challenging" else None
    tables[d] = pd.DataFrame({m: quiet_eval(p, gt) for m, p in methods.items()}).T
    print(f"\n=== {d.upper()} ===")
    display(tables[d].round(4))
    tables[d].round(4).to_csv(f"{TEST_OUT}/table_classical_{d}.csv")


=== LSUI ===


,UCIQE,UIQM,PSNR,SSIM
Raw,0.4264,1.9818,13.2736,0.7118
GrayWorld,0.4532,1.9788,14.9437,0.7647
CLAHE,0.4612,2.5779,13.8718,0.7816
GW+CLAHE,0.4926,2.6683,15.9077,0.8278
DCP,0.5081,2.3452,13.0247,0.6780
UDCP,0.4984,2.4545,11.9981,0.6126
MD-Net,0.5582,2.6557,21.5278,0.8950



=== UIEB ===


,UCIQE,UIQM,PSNR,SSIM
Raw,0.5184,2.5421,17.0433,0.7653
GrayWorld,0.5549,2.5960,16.7225,0.7711
CLAHE,0.5478,2.8760,17.0887,0.7990
GW+CLAHE,0.5827,3.0269,17.0920,0.8013
DCP,0.5600,2.7110,15.3382,0.7285
UDCP,0.5814,2.6831,12.4508,0.6178
MD-Net,0.5957,2.8946,20.3277,0.8715



=== CHALLENGING ===


,UCIQE,UIQM
Raw,0.4743,1.9206
GrayWorld,0.5039,1.9516
CLAHE,0.5071,2.3545
GW+CLAHE,0.5380,2.4498
DCP,0.5102,2.0966
UDCP,0.5317,2.1261
MD-Net,0.5599,2.3277


---
# Section 5 — LPIPS metric
LPIPS is a perceptual distance between the output and the reference (**lower is better**), complementing PSNR and SSIM. It is computed for each method against the reference on LSUI and UIEB only (Challenging has no reference).

In [ ]:
!pip -q install lpips

In [ ]:
import lpips
from torchvision.transforms.functional import to_tensor

lpips_fn = lpips.LPIPS(net="alex", verbose=False).to(device).eval()

def load_lpips_tensor(path, size=None):
    img = Image.open(path).convert("RGB")
    if size: img = img.resize((size, size), Image.BICUBIC)
    return to_tensor(img) * 2 - 1          # LPIPS expects [-1, 1]

@torch.no_grad()
def lpips_folder(pred_dir, gt_dir, bs=16):
    names = sorted(f for f in os.listdir(pred_dir) if f.lower().endswith((".png", ".jpg", ".jpeg")))
    vals = []
    for i in range(0, len(names), bs):
        chunk = names[i:i + bs]
        a = torch.stack([load_lpips_tensor(f"{pred_dir}/{n}", IMG_SIZE) for n in chunk]).to(device)
        b = torch.stack([load_lpips_tensor(f"{gt_dir}/{n}",   IMG_SIZE) for n in chunk]).to(device)
        vals.extend(lpips_fn(a, b).flatten().cpu().tolist())
    return float(np.mean(vals)), len(vals)

lpips_table = {}
for d in ("lsui", "uieb"):
    lpips_table[d] = {m: lpips_folder(p, f"{TEST_OUT}/{d}/references")[0] for m, p in METHOD_DIRS[d].items()}
lp = pd.DataFrame(lpips_table).round(4)
lp.index.name = "LPIPS (lower is better)"
display(lp)
lp.to_csv(f"{TEST_OUT}/table_lpips.csv")

---
# Section 6 — Applied test (similar to what the paper did)
We measure whether enhancement benefits simple vision tasks, on raw images, classical methods, and MD-Net:
- **SIFT keypoints:** number of keypoints (more = more detectable features).
- **SIFT matches vs ref:** number of good matches (ratio test 0.75) with the reference image. A more robust metric because it compares against the reference.
- **Canny edge F1 vs ref:** edge agreement with the reference (one-pixel tolerance).
- **Canny edge density:** edge density. **Warning:** it may increase due to noise amplification, so do not read it alone.

In [ ]:
sift = cv2.SIFT_create()

def _gray(path):
    return cv2.cvtColor(read_rgb(path, IMG_SIZE), cv2.COLOR_RGB2GRAY)

def _edges(gray):
    return cv2.Canny(gray, 100, 200) > 0

def _edge_f1(e_pred, e_ref):
    k = np.ones((3, 3), np.uint8)
    d_ref  = cv2.dilate(e_ref.astype(np.uint8),  k) > 0
    d_pred = cv2.dilate(e_pred.astype(np.uint8), k) > 0
    p = (e_pred & d_ref).sum() / max(e_pred.sum(), 1)
    r = (e_ref & d_pred).sum() / max(e_ref.sum(), 1)
    return 2 * p * r / max(p + r, 1e-9)

bf = cv2.BFMatcher(cv2.NORM_L2)

def app_metrics(pred_dir, ref_dir=None):
    names = sorted(f for f in os.listdir(pred_dir) if f.lower().endswith((".png", ".jpg", ".jpeg")))
    rows = []
    for n in names:
        g = _gray(f"{pred_dir}/{n}")
        kp, des = sift.detectAndCompute(g, None)
        e = _edges(g)
        row = {"SIFT keypoints": len(kp), "Canny density": e.mean()}
        if ref_dir:
            gr = _gray(f"{ref_dir}/{n}")
            _, des_r = sift.detectAndCompute(gr, None)
            good = 0
            if des is not None and des_r is not None and len(des) > 1 and len(des_r) > 1:
                good = sum(1 for m in bf.knnMatch(des, des_r, k=2)
                           if len(m) == 2 and m[0].distance < 0.75 * m[1].distance)
            row["SIFT matches vs ref"] = good
            row["Canny edge F1 vs ref"] = _edge_f1(e, _edges(gr))
        rows.append(row)
    return pd.DataFrame(rows).mean()

app_tables = {}
for d, methods in METHOD_DIRS.items():
    ref = f"{TEST_OUT}/{d}/references" if d != "challenging" else None
    app_tables[d] = pd.DataFrame({m: app_metrics(p, ref) for m, p in methods.items()}).T
    print(f"\n=== {d.upper()} ===")
    display(app_tables[d].round(4))
    app_tables[d].round(4).to_csv(f"{TEST_OUT}/table_application_{d}.csv")


=== LSUI ===


,SIFT keypoints,Canny density,SIFT matches vs ref,Canny edge F1 vs ref
Raw,304.2206,0.0264,251.7711,0.2565
GrayWorld,345.1093,0.0310,225.6907,0.3034
CLAHE,989.9691,0.0936,507.0515,0.6887
GW+CLAHE,1041.7381,0.0983,490.7175,0.6642
DCP,479.9897,0.0435,367.5381,0.4829
UDCP,505.2103,0.0467,370.6371,0.4461
MD-Net,997.1959,0.0791,593.7175,0.7451



=== UIEB ===


,SIFT keypoints,Canny density,SIFT matches vs ref,Canny edge F1 vs ref
Raw,719.6000,0.0697,580.3111,0.6497
GrayWorld,829.7000,0.0867,526.5222,0.6669
CLAHE,1430.8444,0.1433,841.1778,0.8327
GW+CLAHE,1498.5000,0.1582,779.3000,0.7919
DCP,836.7111,0.0826,638.7889,0.7350
UDCP,879.1444,0.0877,626.3889,0.7381
MD-Net,1172.0444,0.1066,802.6889,0.8622



=== CHALLENGING ===


,SIFT keypoints,Canny density
Raw,330.8000,0.0414
GrayWorld,391.7833,0.0516
CLAHE,870.9500,0.0988
GW+CLAHE,910.2833,0.1046
DCP,407.1833,0.0512
UDCP,466.7167,0.0592
MD-Net,838.3833,0.0732


---
# Section 7 — Qualitative comparison (image grids for README and publications)
Requires running Section 4 (defines `METHOD_DIRS` and `read_rgb`). The grid automatically selects **good, average, and failed** images so it doesn't only show the best.

In [ ]:
import matplotlib.pyplot as plt
assert "METHOD_DIRS" in globals() and "read_rgb" in globals(), "Run Section 4 first"

FIG_DIR = f"{TEST_OUT}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
SHOW = ["Raw", "CLAHE", "GW+CLAHE", "MD-Net"]      # Columns (followed by the reference, if available)

def _stem_path(folder, stem):
    for ext in (".png", ".jpg", ".jpeg"):
        p = f"{folder}/{stem}{ext}"
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{folder}/{stem}.*")

def pick_images(dataset, n_good=2, n_mid=1, n_bad=2, seed=0):
    """Sorts the set's images by MD-Net quality (PSNR against the reference, or UIQM when no reference is available)
    and returns [(stem, 'good'/'median'/'failure')]."""
    out_dir = METHOD_DIRS[dataset]["MD-Net"]
    stems = sorted(os.path.splitext(f)[0] for f in os.listdir(out_dir) if f.lower().endswith((".png", ".jpg")))
    ref_dir = f"{TEST_OUT}/{dataset}/references" if dataset != "challenging" else None
    if ref_dir:
        score = {s: M.psnr(read_rgb(_stem_path(out_dir, s), IMG_SIZE), read_rgb(_stem_path(ref_dir, s), IMG_SIZE)) for s in stems}
    else:
        score = {s: M.uiqm(read_rgb(_stem_path(out_dir, s), IMG_SIZE)) for s in stems}
    ranked = sorted(stems, key=lambda s: score[s])                      # from worst to best
    rng = np.random.default_rng(seed)
    top = list(rng.choice(ranked[int(len(ranked) * 0.7):], n_good, replace=False))   # Random from the top 30%
    mid_lo, mid_hi = int(len(ranked) * 0.4), int(len(ranked) * 0.6)
    mid = list(rng.choice(ranked[mid_lo:mid_hi], n_mid, replace=False))
    bad = ranked[:n_bad]                                                  # the actual worst images
    return [(s, "good") for s in top] + [(s, "median") for s in mid] + [(s, "failure") for s in bad]

def make_grid(dataset, picks, methods=SHOW, title=None, dpi=160):
    ref_dir = f"{TEST_OUT}/{dataset}/references" if dataset != "challenging" else None
    cols = list(methods) + (["Reference"] if ref_dir else [])
    fig, axes = plt.subplots(len(picks), len(cols), figsize=(2.6 * len(cols), 2.6 * len(picks)))
    axes = np.atleast_2d(axes)
    for r, (stem, tag) in enumerate(picks):
        for c, col in enumerate(cols):
            folder = ref_dir if col == "Reference" else METHOD_DIRS[dataset][col]
            ax = axes[r, c]
            ax.imshow(read_rgb(_stem_path(folder, stem), IMG_SIZE))
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
            if r == 0: ax.set_title(col, fontsize=11)
        axes[r, 0].set_ylabel(f"{tag}\n{stem[:10]}", fontsize=9, rotation=0, ha="right", va="center", labelpad=8)
    if title: fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    path = f"{FIG_DIR}/grid_{dataset}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print("saved:", path)

In [ ]:
for d, t in (("lsui", "LSUI test"), ("uieb", "UIEB test"), ("challenging", "UIEB Challenging (no reference)")):
    make_grid(d, pick_images(d), title=t)